# L4 · Notebook 03 — GPI 锯齿可视化（PI / VI / 截断 PI 三条路径）

**对应 PPT**：`L4-DP.tex` ★ GPI 经典图示帧（line 961）

## 教学目标

把广义策略迭代 GPI 抽象为 $(V, \pi)$ 平面上的迭代过程：
- 评估步：$V \to V^\pi$（往"$V = V^\pi$"曲线方向走）
- 改进步：$\pi \to \pi'$（往"$\pi = \text{greedy}(V)$"曲线方向走）

三种算法对应三种锯齿模式：
1. **PI**（完整评估）：长竖线（评估到底）+ 短横线（改进）
2. **VI**（评估只 1 步）：等价于 $V \to T^*V$，无显式 $\pi$ 步
3. **截断 PI**（评估 $m$ 步）：中等竖线 + 横线，介于两者之间

把三条路径画在同一个 $(V, \pi)$ 平面上 → 直观看清"评估深度"作为 GPI 的可调旋钮。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'L1-mdp-foundations')))
from grid_world import ACTIONS, GridConfig, GridWorld

## 1. 用 $(V, \pi)$ 的两个标量轴表示状态

GPI 平面是 $|\S|$ 维 $\times |\A|^{|\S|}$ 维高维空间，难以可视化。我们用两个标量代理：

- 横轴 $V$ 代理：$\bar V = \frac{1}{|\S|}\sum_s V(s)$（状态价值平均）
- 纵轴 $\pi$ 代理：$d(\pi, \pi^*) = $ 与最优策略不一致的状态数（贴近 0 = 接近最优）

这样每个 $(V, \pi)$ 对在二维平面上对应一个点；GPI 轨迹连成锯齿/曲线。

In [ ]:
env = GridWorld(GridConfig(gamma=0.9))
states = list(env.all_states())
nS = len(states)
idx = {s: i for i, s in enumerate(states)}

def pe_step(V, policy):
    V_new = np.zeros_like(V)
    for i, s in enumerate(states):
        a = policy[s]
        s_next, r, _ = env.step(s, a)
        V_new[i] = r + env.cfg.gamma * V[idx[s_next]]
    return V_new

def pe_full(V, policy, tol=1e-10):
    for _ in range(2000):
        V_new = pe_step(V, policy)
        if np.max(np.abs(V_new - V)) < tol:
            return V_new
        V = V_new
    return V

def greedy(V):
    pol = {}
    for i, s in enumerate(states):
        bq, ba = -np.inf, ACTIONS[0]
        for a in ACTIONS:
            s_next, r, _ = env.step(s, a)
            q = r + env.cfg.gamma * V[idx[s_next]]
            if q > bq:
                bq, ba = q, a
        pol[s] = ba
    return pol

def vi_step(V):
    V_new = np.zeros_like(V)
    for i, s in enumerate(states):
        best = -np.inf
        for a in ACTIONS:
            s_next, r, _ = env.step(s, a)
            q = r + env.cfg.gamma * V[idx[s_next]]
            if q > best:
                best = q
        V_new[i] = best
    return V_new

# 先算最优策略作为锚
V_init = np.zeros(nS)
V_star = V_init.copy()
for _ in range(2000):
    V_new = vi_step(V_star)
    if np.max(np.abs(V_new - V_star)) < 1e-12:
        V_star = V_new
        break
    V_star = V_new
pi_star = greedy(V_star)
print(f'V*: mean={V_star.mean():.3f}, max={V_star.max():.3f}')

def diff_to_optimal(policy):
    return sum(1 for s in states if policy[s] != pi_star[s])

## 2. 三条算法轨迹

起点：$V_0 \equiv 0$，$\pi_0 = $ 全 "stay"。

In [ ]:
def run_pi(eval_steps):
    """运行 (截断) 策略迭代；记录 (V_mean, π_diff) 在每个 sub-step。
    eval_steps=None 表示完整 PE 直到收敛。"""
    V = np.zeros(nS)
    policy = {s: 'stay' for s in states}
    trajectory = [(V.mean(), diff_to_optimal(policy))]
    
    for outer in range(20):
        # 评估阶段（连续 eval_steps 次 PE 更新；记录中间点显示"竖线"）
        if eval_steps is None:
            V = pe_full(V, policy)
            trajectory.append((V.mean(), diff_to_optimal(policy)))
        else:
            for _ in range(eval_steps):
                V = pe_step(V, policy)
                trajectory.append((V.mean(), diff_to_optimal(policy)))
        
        # 改进阶段（V 不变，policy 跳变 → 显示"横线"）
        new_pol = greedy(V)
        trajectory.append((V.mean(), diff_to_optimal(new_pol)))
        if all(new_pol[s] == policy[s] for s in states):
            break
        policy = new_pol
    return np.array(trajectory)

def run_vi():
    """VI 等价于 "PE 1 步 + 隐式改进"。记录每步 (V_mean, π_diff)。"""
    V = np.zeros(nS)
    trajectory = [(V.mean(), diff_to_optimal({s: 'stay' for s in states}))]
    for k in range(200):
        V = vi_step(V)
        pol = greedy(V)
        trajectory.append((V.mean(), diff_to_optimal(pol)))
        if k > 0 and np.allclose(V, V_star, atol=1e-6):
            break
    return np.array(trajectory)

traj_pi_full = run_pi(eval_steps=None)
traj_pi_trunc = run_pi(eval_steps=3)
traj_vi = run_vi()
print(f'完整 PI 轨迹长度: {len(traj_pi_full)}')
print(f'截断 PI(m=3) 长度: {len(traj_pi_trunc)}')
print(f'VI 长度: {len(traj_vi)}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for traj, label, color, marker in [
    (traj_pi_full,  '完整 PI (eval=∞)',      'C0', 'o'),
    (traj_pi_trunc, '截断 PI (eval=3)',       'C1', 's'),
    (traj_vi,       'VI (eval=1, 隐式 π 跳)', 'C2', '^'),
]:
    ax.plot(traj[:, 0], traj[:, 1], '-', color=color, alpha=0.6, linewidth=1)
    ax.plot(traj[:, 0], traj[:, 1], marker, color=color, markersize=5, label=label)

ax.scatter([V_star.mean()], [0], color='red', s=200, marker='*', zorder=5, label='最优 $(V^*, \\pi^*)$')
ax.set_xlabel(r'$\bar V = \frac{1}{|S|}\sum_s V(s)$（V 远离/接近 V*）')
ax.set_ylabel(r'$d(\pi, \pi^*)$ = 与最优策略不一致的状态数')
ax.set_title('GPI 锯齿图：评估深度作为 PI ↔ VI 谱系的可调旋钮')
ax.legend(loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figures/gpi_zigzag.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. 课堂诊断小结

**观察轨迹形态**：

| 算法 | 评估深度 | 单步代价 | 外层迭代次数 |
|---|---|---|---|
| 完整 PI | $\infty$（评到不动点）| 高 | 极少（通常 3-6 轮） |
| 截断 PI (m=3) | 3 | 中 | 中 |
| VI | 1 | 低（每步 O$(\|\mathcal S\|^2\|\mathcal A\|)$）| 多（$\gamma^k$ 衰减）|

**核心 insight**：GPI 不是某个具体算法，而是 "评估 + 改进" 反复进行的统一框架——评估深度是一个连续旋钮，PI 和 VI 是该旋钮的两端。

## 思考题

1. 把 eval_steps 改成 1，截断 PI 是否退化为 VI？两者轨迹是否完全重合？
2. GPI 在 L7 TD / L10 AC 中怎么体现？（提示：策略评估 → critic 学 $V^\pi$；策略改进 → actor 沿 $\nabla J$ 上升）
3. 完整 PI 每外层迭代 $V$ 跳变幅度大，VI 每步 $V$ 跳变小但策略也常变——两种轨迹哪个对**函数逼近**更友好？为什么？